# `new_uc` — preset notebooks and custom methods

This demo builds a small team-flavored layer on top of unichart:

1. **`UcNotebook`** — a subclass of `UnichartNotebook` that carries your own
   methods. The example here is `load_elog`, a loader for a fictional `.elog`
   engine-log format (metadata header + CSV body) that no generic reader knows.
2. **`new_uc(preset, ...)`** — a factory that returns a ready-configured
   `UcNotebook`. Presets can be **layered** (`new_uc(["report", "dark"])`)
   and any preset knob can be **overridden inline**
   (`new_uc("dark", figsize=(9, 5))`).

Presets bundled in this demo:

| Preset | Look | Key settings |
|---|---|---|
| `default` | unichart out-of-the-box | matplotlib style, light mode |
| `dark` | dashboard on a dark background | dark mode, plotly style, vivid palette |
| `report` | static, document-friendly | static PNG output, muted palette, footer |
| `presentation` | big-screen slides | large fonts, thick lines, big markers |

Everything here uses the public `UnichartNotebook` API, so any preset value can
still be changed afterwards on the returned notebook.

## Setup — imports and sample data

In [ ]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook


def sample_data(seed=7):
    """Three 'engine runs': an RPM sweep with a few sensor channels each."""
    rng = np.random.default_rng(seed)
    rows = []
    for i, eng in enumerate(["Engine A", "Engine B", "Engine C"]):
        rpm = np.linspace(1000, 6000, 40)
        temp = 60 + 0.012 * rpm + 8 * i + rng.normal(0, 2.5, rpm.size)
        pressure = 14 + 0.004 * rpm + 1.5 * i + rng.normal(0, 0.8, rpm.size)
        thrust = 0.9 * (rpm / 1000) ** 1.8 + 2 * i + rng.normal(0, 0.6, rpm.size)
        rows.append(pd.DataFrame({
            "ENGINE": eng, "RPM": rpm, "Temp": temp,
            "Pressure": pressure, "Thrust": thrust,
        }))
    return pd.concat(rows, ignore_index=True)


df = sample_data()
df.head()

## The custom notebook class — where your methods live

Subclassing `UnichartNotebook` is the natural home for anything your team does
repeatedly that unichart can't know about. The example: our (fictional) test rig
writes **`.elog`** files — `!`-prefixed metadata lines, then plain CSV:

```
! ENGINE: Engine A
! TEST: Hot-day takeoff
! DATE: 2026-08-12
RPM,Temp,Pressure
1200,75.31,18.92
...
```

`load_elog` parses the header, loads the body as a dataset titled from the
metadata, injects the metadata as columns, and registers them as hover
display parms — so every point on a plot tells you which test it came from.

In [ ]:
class UcNotebook(UnichartNotebook):
    """UnichartNotebook plus our team's custom loaders and helpers."""

    _ELOG_META_PARMS = ("TEST", "DATE")   # metadata keys surfaced in hover boxes

    def load_elog(self, path):
        """Load one .elog engine log (``!`` metadata header + CSV body).

        Returns the new Dataset. The set is titled "<ENGINE> — <TEST>", the
        metadata is added as constant columns, and TEST/DATE become hover
        display parms.
        """
        meta = {}
        with open(path) as f:
            for line in f:
                if not line.startswith("!"):
                    break
                key, _, val = line[1:].partition(":")
                meta[key.strip().upper()] = val.strip()

        df = pd.read_csv(path, comment="!")

        # Metadata -> constant columns, so it can appear in hovers and queries.
        for key in self._ELOG_META_PARMS:
            if key in meta:
                df[key] = meta[key]

        title = meta.get("ENGINE", os.path.basename(path))
        if "TEST" in meta:
            title = f"{title} — {meta['TEST']}"

        self.load_df(df, title=title)
        ds = self.sets[-1]
        ds.file_path = os.path.abspath(path)
        parms = [k for k in self._ELOG_META_PARMS if k in meta]
        if parms:
            self.set_display_parms(ds, parms)
        return ds

## The `new_uc` factory

`PRESETS` is a plain, user-editable dict. `new_uc` improves on a one-preset
factory in three ways:

- **Layering** — pass a list of names and later presets override earlier ones,
  with dict-valued knobs (`font_sizes`, `default_format`, `static_images`)
  merged key-by-key: `new_uc(["report", "dark"])` is a dark static report.
- **Inline overrides** — any preset knob works as a keyword argument and wins
  over the preset: `new_uc("dark", figsize=(9, 5))`.
- **Data-aware** — remaining keyword arguments go to `load_df`, and loading
  happens *after* styling so datasets pick up the preset palette/markers.

In [ ]:
PRESETS = {
    # unichart out-of-the-box: matplotlib look, light mode. Kept explicit so the
    # registry documents what "default" means.
    "default": {
        "plot_style": "matplotlib",
        "darkmode": False,
    },

    # Dark dashboard: plotly house style in dark mode with a vivid palette.
    "dark": {
        "plot_style": "plotly",
        "darkmode": True,
        "color_map": ["#00E5FF", "#FF4081", "#FFD740", "#69F0AE", "#B388FF"],
        "marker_map": ["o", "D", "s", "^", "v"],
        "font_sizes": {"suptitle": 22, "legend": 13, "axes_title": 15},
    },

    # Report/publication: flat PNGs keep the .ipynb (and exported HTML) small,
    # a muted palette prints well, and every figure carries a footer.
    "report": {
        "plot_style": "matplotlib",
        "darkmode": False,
        "static_images": {"enabled": True, "scale": 2},
        "copy_buttons": False,
        "color_map": ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"],
        "default_format": {"markersize": 5, "linewidth": 1.2},
        "figsize": (10, 6),
        "footer": "unichart report preset — data is synthetic",
    },

    # Presentation: readable from the back of the room.
    "presentation": {
        "plot_style": "matplotlib",
        "darkmode": False,
        "color_map": ["#E63946", "#457B9D", "#2A9D8F", "#F4A261"],
        "default_format": {"markersize": 11, "linewidth": 3.5, "edgewidth": 2},
        "font_sizes": {"all": 16, "suptitle": 28, "axes_title": 20, "legend": 18},
        "figsize": (14, 8),
    },
}

# Every knob a preset (or an inline override) may set. Anything else passed to
# new_uc as a keyword is routed to load_df.
_PRESET_KEYS = frozenset({
    "plot_style", "darkmode", "color_map", "marker_map", "default_format",
    "figsize", "font_sizes", "footer", "static_images", "copy_buttons",
})

# Dict-valued knobs layer key-by-key instead of wholesale replacement, so
# ["report", "dark"] keeps report's static_images while taking dark's fonts.
_MERGED_KEYS = ("default_format", "font_sizes", "static_images")


def _layer(cfg, layer):
    """Fold one preset layer (or the inline overrides) into cfg."""
    for key, value in layer.items():
        if key in _MERGED_KEYS and isinstance(value, dict):
            cfg[key] = {**cfg.get(key, {}), **value}
        else:
            cfg[key] = value


def new_uc(preset="default", data=None, **kwargs):
    """Return a new UcNotebook configured by preset(s).

    Parameters
    ----------
    preset : str or list of str
        A key of ``PRESETS``, or a list of keys layered left-to-right (later
        presets win; dict-valued knobs merge key-by-key).
    data : DataFrame, optional
        If given, loaded via ``nb.load_df(data, ...)`` *after* the presets are
        applied, so datasets inherit the preset's palette/markers.
    **kwargs
        Keys named in ``_PRESET_KEYS`` (e.g. ``figsize=``, ``darkmode=``)
        override the preset; everything else (e.g. ``set_name_column=``) is
        passed to ``load_df``.
    """
    names = [preset] if isinstance(preset, str) else list(preset)
    cfg = {}
    for name in names:
        if name not in PRESETS:
            raise ValueError(f"Unknown preset {name!r}. "
                             f"Available: {sorted(PRESETS)}")
        _layer(cfg, PRESETS[name])

    overrides = {k: kwargs.pop(k) for k in list(kwargs) if k in _PRESET_KEYS}
    _layer(cfg, overrides)
    if kwargs and data is None:
        raise TypeError(f"load arguments {sorted(kwargs)} given without data=")

    nb = UcNotebook()

    # Overall look first: plot style installs its own color map / format
    # defaults, so preset-specific palettes must be applied after it.
    if "plot_style" in cfg:
        nb.set_plot_style(cfg["plot_style"])
    nb.toggle_darkmode(cfg.get("darkmode", False))

    if "color_map" in cfg:
        nb.color_map = list(cfg["color_map"])
    if "marker_map" in cfg:
        nb.marker_map = list(cfg["marker_map"])
    if "default_format" in cfg:
        nb.set_default_format(**cfg["default_format"])
    if "figsize" in cfg:
        nb.set_default_format(figsize=cfg["figsize"])
    if "font_sizes" in cfg:
        nb.set_font_sizes(**cfg["font_sizes"])
    if "footer" in cfg:
        nb.footer = cfg["footer"]
    if "static_images" in cfg:
        nb.set_static_images(**cfg["static_images"])
    if "copy_buttons" in cfg:
        nb.set_copy_buttons(cfg["copy_buttons"])

    if data is not None:
        nb.load_df(data, **kwargs)
    return nb

## 1. `default` — the baseline

One call gives a notebook with data already loaded and split into per-engine datasets.

In [ ]:
nb = new_uc("default", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb.plot(x="RPM", y=["Temp", "Pressure"], suptitle="default preset")

## 2. `dark` — same data, different environment

Swapping the preset name is the only change: dark plotly template, vivid palette,
its own marker sequence and font sizes.

In [ ]:
nb_dark = new_uc("dark", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb_dark.plot(x="RPM", y=["Temp", "Pressure"], suptitle="dark preset")

## 3. `report` — static, document-friendly figures

This preset turns on static PNG output (requires `kaleido`; falls back to interactive
if it isn't installed), uses a muted print palette, and pins a footer on every figure.

In [ ]:
nb_report = new_uc("report", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb_report.plot(x="RPM", y="Thrust", suptitle="report preset")

## 4. `presentation` — readable from the back of the room

In [ ]:
nb_pres = new_uc("presentation", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb_pres.plot(x="RPM", y="Temp", suptitle="presentation preset")

## 5. Layer presets and override inline

`["report", "dark"]` folds `dark` on top of `report`: the result keeps report's
static output, footer and small markers while taking dark's palette and theme —
a dark static report. The `figsize=` keyword then overrides both presets.

In [ ]:
nb_mix = new_uc(["report", "dark"], data=df,
                set_name_column="ENGINE", set_idx_column="ENGINE",
                figsize=(9, 5))                     # inline override beats both presets

print("darkmode:", nb_mix.darkmode,
      "| static:", nb_mix.static_images,
      "| figsize:", nb_mix.figsize,
      "| footer:", repr(nb_mix.footer))
nb_mix.plot(x="RPM", y="Thrust", suptitle="report + dark, figsize overridden")

## 6. The custom method in action — loading `.elog` files

First fabricate a few `.elog` files (this stands in for whatever your rig
produces), then load them with the method that ships on every `new_uc` notebook.
Hover over the points: TEST and DATE ride along from the file headers.

In [ ]:
import tempfile

def write_demo_elog(path, engine, test, date, bias, seed):
    """Fabricate one .elog file: '!' metadata header + CSV body."""
    rng = np.random.default_rng(seed)
    rpm = np.linspace(1200, 5800, 30)
    temp = 62 + 0.011 * rpm + bias + rng.normal(0, 2, rpm.size)
    press = 14 + 0.004 * rpm + 0.5 * bias + rng.normal(0, 0.5, rpm.size)
    lines = [f"! ENGINE: {engine}", f"! TEST: {test}", f"! DATE: {date}",
             "RPM,Temp,Pressure"]
    lines += [f"{r:.0f},{t:.2f},{p:.2f}" for r, t, p in zip(rpm, temp, press)]
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")

log_dir = tempfile.mkdtemp(prefix="uc_elogs_")
runs = [("Engine A", "Hot-day takeoff", "2026-08-12", 0,  1),
        ("Engine A", "Cold soak",       "2026-08-14", -6, 2),
        ("Engine B", "Hot-day takeoff", "2026-08-13", 9,  3)]
log_paths = []
for eng, test, date, bias, seed in runs:
    p = os.path.join(log_dir, f"{eng.replace(' ', '_')}_{seed}.elog")
    write_demo_elog(p, eng, test, date, bias, seed)
    log_paths.append(p)

print(open(log_paths[0]).read().splitlines()[0:5])   # peek at the format

In [ ]:
nb_logs = new_uc("dark")
for p in log_paths:
    nb_logs.load_elog(p)

# titles came from the metadata; TEST/DATE are hover display parms
for ds in nb_logs.sets:
    print(ds.title_format, "| display_parms:", ds.display_parms)

nb_logs.plot(x="RPM", y="Temp", suptitle="datasets loaded from .elog files")

## 7. Presets are starting points, not straitjackets

Register your own preset by adding a dict entry — it composes with everything
above (layering, overrides, `load_elog`) for free.

In [ ]:
PRESETS["mono"] = {
    "plot_style": "matplotlib",
    "color_map": ["#222222", "#666666", "#AAAAAA"],
    "marker_map": ["o", "s", "^"],
    "default_format": {"markersize": 6, "linewidth": 2},
}

nb_mono = new_uc("mono", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb_mono.plot(x="RPM", y="Pressure", suptitle="custom 'mono' preset")